# Supervised Fine-Tuning (SFT) with Serverless Customization on SageMaker AI

## Lab 2 – Fine-Tune an LLM with Serverless Customization

**Prerequisite:** Complete **Lab 1** (`1-prepare-data.ipynb`) before running this notebook. The `Multilingual-Thinking-sft-train` and `Multilingual-Thinking-sft-val` datasets must be registered in the SageMaker AI Registry.

This is the second of four interconnected labs:

| Lab | Notebook | What you'll do |
|-----|----------|----------------|
| **Lab 1** | `1-prepare-data.ipynb` | Prepare and register the dataset |
| **Lab 2** | `2-fine-tune-llm.ipynb` ← *you are here* | Submit a serverless LoRA fine-tuning job and register the result in the Model Registry |
| **Lab 3** | `3-evaluation.ipynb` | Evaluate the fine-tuned model with LLM-as-Judge |
| **Lab 4** | `4-deployment.ipynb` | Deploy the merged model to a real-time SageMaker endpoint |

### What you'll do in this lab

1. **Select a base model** from SageMaker JumpStart — the starting point for fine-tuning
2. **Create a Model Package Group** in the SageMaker Model Registry — a versioned container for your fine-tuned models
3. **Configure the `SFTTrainer`** with LoRA and the datasets registered in Lab 1
4. **Inspect and override hyperparameters** — understanding the knobs that control fine-tuning quality
5. **Submit the serverless fine-tuning job** and track its progress

The fine-tuned model registered at the end of this lab becomes the artifact used in Labs 3 and 4.

### Prerequisites

Before running this notebook, confirm that:
- **Lab 1 is complete** — the three dataset splits are registered in the SageMaker AI Registry
- Your IAM execution role has `AmazonSageMakerFullAccess` and S3 read/write permissions

***

### Step 1 – Choose a base model

The base model used across all labs is configured in [`config.py`](config.py), defaulting to **NVIDIA Nemotron 3 Nano 30B** (`nvidia/nemotron-3-nano-30b-a3b`).

**What is NVIDIA Nemotron 3 Nano 30B?**
Nemotron 3 Nano 30B is a mixture-of-experts (MoE) model with 30 billion total parameters and ~3 billion active parameters per forward pass. The MoE architecture activates only a subset of model parameters for each token, making it significantly more efficient than dense models of comparable quality. It is available through [SageMaker JumpStart](https://docs.aws.amazon.com/sagemaker/latest/dg/studio-jumpstart.html).

**Also available: NVIDIA Nemotron 3 Super 120B**
Nemotron 3 Super is also available for fine-tuning through SageMaker JumpStart serverless model customization. It is a 120B total parameter model (12B active) — the same MoE architecture as Nano but with significantly more capacity, making it a strong alternative if you want a larger, more capable base for your fine-tuning runs.

**Why fine-tune rather than prompt?**
Prompting a base model is fast but has limits — it requires verbose system instructions at every call and may not reliably follow complex format requirements (like our `<think>` tag structure). Fine-tuning *bakes* the target behavior into the model weights, producing consistent outputs without elaborate prompts at inference time.

Want to try a different model? Update `BASE_MODEL_ID` in `config.py` — all notebooks pick up the change automatically. The cell below lists all JumpStart models that support customization.

In [ ]:
import boto3
from config import BASE_MODEL_ID

# Retrieve all JumpStart models that support customization (fine-tuning)
sm = boto3.client("sagemaker")
models = []
kwargs = {"HubName": "SageMakerPublicHub", "HubContentType": "Model", "MaxResults": 100}
while True:
    response = sm.list_hub_contents(**kwargs)
    for item in response["HubContentSummaries"]:
        keywords = item.get("HubContentSearchKeywords", [])
        if "@capability:customization" in keywords:
            models.append(item["HubContentName"])
    if "NextToken" in response:
        kwargs["NextToken"] = response["NextToken"]
    else:
        break

models.sort()
print(f"Current model: {BASE_MODEL_ID}\n")
print(f"Available models ({len(models)}):")
print("\n".join(models))

In [ ]:
%load_ext autoreload
%autoreload 2

***


### Step 2 – Set up the SageMaker session

#### Setup and dependencies

We re-establish the SageMaker session, retrieve the execution role, and load the registered datasets from Lab 1. Fine-tuning outputs will be written to the session's default S3 bucket under a prefix named after the base model.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
import os
from sagemaker.ai_registry.dataset import DataSet
from config import BASE_MODEL_ID

base_model_id = BASE_MODEL_ID

training_dataset = DataSet.get(name="Multilingual-Thinking-sft-train")
val_dataset = DataSet.get(name="Multilingual-Thinking-sft-val")

if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{base_model_id}"
else:
    output_path = f"s3://{bucket_name}/{base_model_id}"

# os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = (
#     f"https://mlflow.sagemaker.{sess.boto_region_name}.app.aws"
# )

***

### Step 3 – Create a Model Package Group

A **Model Package Group** is a container in the [SageMaker Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry.html) that holds successive versions of a model. Think of it like a Git repository for your model weights — each fine-tuning run registers a new version, preserving the full history.

Benefits of using a Model Package Group:
- **Lineage** — SageMaker records which dataset, training job, and hyperparameters produced each model version
- **Comparison** — View metrics side-by-side across model versions directly in the SageMaker console
- **Clean handoff** — Lab 3 (evaluation) and Lab 4 (deployment) retrieve the latest model from this group by name, without hardcoding ARNs


In [ ]:
import hashlib
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

MAX_MPG_NAME_LENGTH = 63 # This is a character limit for a Model Package Group name. 
suffix = "-sft-mpg"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    # reserve room for the suffix, a hyphen separator, and the 6-char hash
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    truncated = base_model_id[:keep].rstrip("-")
    model_package_group_name = f"{truncated}-{digest}{suffix}"
else:
    model_package_group_name = candidate

try:
    model_package_group = ModelPackageGroup.get(
        model_package_group_name=model_package_group_name
    )
    print(f"Model Package Group already exists: {model_package_group_name}")
except ClientError:
    model_package_group = ModelPackageGroup.create(
        model_package_group_name=model_package_group_name,
        model_package_group_description="Store models from SageMaker serverless SFT customization",
    )
    print(f"Created Model Package Group: {model_package_group_name}")

***

### Step 4 – Configure the serverless SFT job

#### What is serverless fine-tuning?

[Amazon SageMaker AI serverless model customization](https://docs.aws.amazon.com/sagemaker/latest/dg/serverless-customization.html) lets you submit a high-level fine-tuning job without manually provisioning or managing GPU instances. You specify *what* you want (a base model, a technique, a dataset) and SageMaker handles the infrastructure — instance selection, distributed training setup, and automatic model registration.

You pay only for compute actually consumed during training, with no idle instance time.

#### Available customization techniques

SageMaker AI serverless model customization supports five techniques. The table below shows which are available for Nemotron 3 Nano 30B:

| Technique | What it does | Nano 30B | Super 120B |
|-----------|-------------|:--------:|:----------:|
| **SFT** — Supervised Fine-Tuning | Trains on labeled prompt-completion pairs to learn a target behavior or format | ✓ | ✓ |
| **RLVR** — RL with Verifiable Rewards | Uses a code-based reward function to score outputs against ground-truth criteria (e.g., math correctness) | ✓ | ✓ |
| **RLAIF** — RL with AI Feedback | Uses an LLM judge as the reward signal — suited for open-ended tasks without hard verifiers | ✓ | ✓ |
| **DPO** — Direct Preference Optimization | Trains on ranked response pairs to align with human preferences — no reward model needed | — | — |
| **Multi-turn RL** | Extends RL to multi-step conversations, rewarding the model over a full dialogue | — | — |

This workshop uses **SFT** — the most common starting point for adapting a model to a new task or output format.

#### Why LoRA (Low-Rank Adaptation)?

We use `TrainingType.LORA`, a parameter-efficient fine-tuning (PEFT) technique that:

- **Freezes** the base model's billions of pre-trained parameters
- Injects small **low-rank adapter matrices** at key transformer layers (typically the attention projections)
- Trains *only* the adapter matrices — a fraction of a percent of total parameters

This makes fine-tuning dramatically cheaper and faster than full-weight training, while achieving comparable domain adaptation quality. At the end of training, the adapter is **merged** back into the base model weights for efficient deployment (see Lab 4).


In [ ]:
from sagemaker.train.common import TrainingType
from sagemaker.train.sft_trainer import SFTTrainer

#### The SFTTrainer

`SFTTrainer` is a high-level class in the SageMaker Python SDK that abstracts away the infrastructure required to run a serverless supervised fine-tuning job. Rather than manually configuring a training script, instance types, or distributed training setup, you describe *what* you want to fine-tune and the SDK handles the rest.

Key parameters:

| Parameter | What it sets |
|-----------|-------------|
| `model` | The base model ID from SageMaker JumpStart |
| `training_type` | The fine-tuning method — `TrainingType.LORA` for LoRA adapters |
| `model_package_group` | The Model Registry group where the fine-tuned model will be registered |
| `training_dataset` / `validation_dataset` | The versioned datasets registered in Lab 1 |
| `s3_output_path` | S3 location for model artifacts and checkpoints |
| `accept_eula` | Acknowledges the base model's end-user license agreement |
| `base_job_name` | Prefix for the training job name (truncated to fit SageMaker's 63-character limit) |



In [ ]:
# SageMaker truncates the training job name to 63 chars *after* appending a
# timestamp. BASE_MODEL_ID is long enough that, left to the SDK default, the
# timestamp gets chopped off entirely and every run collides on the same name.
# Pass a short base_job_name that leaves room for the "-YYYYMMDDHHMMSS" suffix
# so each run gets a unique job name.
MAX_JOB_NAME_LENGTH = 63
TIMESTAMP_LENGTH = 15  # "-YYYYMMDDHHMMSS"
base_job_name = f"{base_model_id}-sft"[: MAX_JOB_NAME_LENGTH - TIMESTAMP_LENGTH].rstrip("-")

trainer = SFTTrainer(
    model=base_model_id,
    training_type=TrainingType.LORA,
    model_package_group=model_package_group_name,
    training_dataset=training_dataset,
    validation_dataset=val_dataset,
    s3_output_path=output_path,
    sagemaker_session=sess,
    role=role,
    accept_eula=True,
    base_job_name=base_job_name,
)

#### Step 4a – Inspect default hyperparameters

Each base model + customization technique combination ships with a tuned **recipe** of default hyperparameters chosen by the model provider. Print these defaults before overriding anything — they're a useful reference for understanding what values are sensible for this model and technique.

In [ ]:
from rich import print as rprint
from rich.pretty import pprint

print("Default Finetuning options:")
pprint(trainer.hyperparameters.to_dict())

#### Step 4b – Override selected hyperparameters

We override a targeted set of values for this workshop run. Here's what each controls and why:

| Hyperparameter | Value | What it controls |
|---------------|-------|-----------------|
| `learning_rate` | `0.0002` | Step size during gradient descent. Too high → unstable training; too low → slow convergence |
| `global_batch_size` | `128` | Total examples processed per gradient update, across all GPUs |
| `max_epochs` | `10` | Number of complete passes through the training data |
| `warmup_steps` | `6` | Gradually ramps the learning rate from 0 to its target value at the start — prevents large updates from destabilizing early training |
| `weight_decay` | `0.05` | L2 regularization to penalize large weights and reduce overfitting |
| `lora_rank` | `32` | Dimensionality of the LoRA adapter matrices. Higher rank = more expressive adapters but more trainable parameters |
| `lora_alpha` | `64` | Scaling factor applied to LoRA updates. Typically set to `2 × lora_rank` |

<br/>

> **Tip:** Leave all other hyperparameters at their recipe defaults unless you have a specific reason to change them. Over-tuning can lead to overfitting or unstable training.

In [ ]:
trainer.hyperparameters.learning_rate = 0.0002
trainer.hyperparameters.global_batch_size = 128
trainer.hyperparameters.max_epochs = 10
trainer.hyperparameters.warmup_steps = 6
trainer.hyperparameters.weight_decay = 0.05
trainer.hyperparameters.lora_rank = 32
trainer.hyperparameters.lora_alpha = 64

In [ ]:
print("\nModified/user defined options:")
pprint(trainer.hyperparameters.to_dict())

### Kick off the training job

The cell below submits the fine-tuning job asynchronously (`wait=False`), so it will return immediately without waiting for training to complete. Training typically takes **20 minutes** with the chosen dataset and hyperparameters, but will depend on the model and dataset size.

Once the job is submitted, you can track its progress in two ways:

1. **AWS Console** — Navigate to **SageMaker AI > Training > Training jobs** and search for the job name.
2. **SageMaker SDK** — Use the status-check cell further below to poll the job status programmatically.

In [ ]:
from rich import print as rprint
from rich.pretty import pprint

training_job = trainer.train(wait=False)

TRAINING_JOB_NAME = training_job.training_job_name

pprint(training_job)

In [ ]:
# Check the training job status
from sagemaker.core.resources import TrainingJob

response = TrainingJob.get(training_job_name=TRAINING_JOB_NAME)
print(f"Status: {response.training_job_status}")
print(f"Secondary: {response.secondary_status}")

#### *(Optional)* Poll until training completes

The cell below loops every 60 seconds and prints the current job status until training finishes or fails. Run it if you want to stay in the notebook rather than switching to the AWS console.

> **Note:** You can interrupt the loop at any time with the **Stop** button — the training job will continue running in the background.

In [ ]:
import time
from datetime import datetime
from sagemaker.core.resources import TrainingJob

POLL_INTERVAL_SECONDS = 30
TERMINAL_STATUSES = {"Completed", "Failed", "Stopped"}  # Stopping is transient — loop continues until Stopped

print(f"Polling job: {TRAINING_JOB_NAME}")
print("-" * 50)

while True:
    response = TrainingJob.get(training_job_name=TRAINING_JOB_NAME)
    status = response.training_job_status
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] Status: {status}")

    if status in TERMINAL_STATUSES:
        if status == "Completed":
            print("\nTraining complete. Proceed to the next cell.")
        else:
            print(f"\nJob ended with status: {status}. Check the SageMaker console for details.")
        break

    time.sleep(POLL_INTERVAL_SECONDS)